# Lumbar Disc Bulging Classification

요추 MRI(T2) 영상에서 추간판 팽윤(Disc Bulging)을 이진 분류하는 프로젝트.

## 데이터셋 준비 (최초 1회)

[SPIDER Dataset (Kaggle)](https://www.kaggle.com/datasets/zofiaknapiska/spider-lumbar-spine-segmentation-in-mr-images)을
다운로드해 아래 구조로 배치한다:

```
data/
├── images/images/            # T2 MRI 볼륨 (.mha)
├── masks/masks/              # 세그멘테이션 마스크 (.mha)
├── overview.csv
└── radiological_gradings.csv
```

또는 kagglehub로 자동 다운로드:
```python
import kagglehub
path = kagglehub.dataset_download("zofiaknapiska/spider-lumbar-spine-segmentation-in-mr-images")
```

# 0. 환경 설정 및 공통 임포트

In [ ]:
import os
import random
import datetime
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from scipy.ndimage import zoom as ndimage_zoom
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

plt.rcParams['figure.dpi'] = 100

In [ ]:
# 데이터셋 루트 경로
dataset_base_path = "data"

# 디바이스 설정
device = torch.device("mps" if torch.backends.mps.is_available()
                       else "cuda" if torch.cuda.is_available()
                       else "cpu")
print(f"Device: {device}")

# 1. 데이터 로드

## Overview CSV

In [ ]:
csv_path = os.path.join(dataset_base_path, "overview.csv")
overview = pd.read_csv(csv_path)
overview.head()

In [ ]:
overview.info()

In [ ]:
overview.describe()

## Radiological Gradings CSV

In [ ]:
csv_path = os.path.join(dataset_base_path, "radiological_gradings.csv")
df_gradings = pd.read_csv(csv_path)
df_gradings.head(10)

In [ ]:
df_gradings.info()

In [ ]:
df_gradings.describe()

# 2. MHA 이미지·마스크 로드

In [ ]:
# Refined approach to find image and mask files based on directory structure
image_dir = os.path.join(dataset_base_path, "images", "images") # Adjust if structure is different
mask_dir = os.path.join(dataset_base_path, "masks", "masks") # Adjust if structure is different

image_files_map = {}
mask_files_map = {}

for f in os.listdir(image_dir):
    if f.endswith('.mha'):
        # Extract a unique identifier (e.g., '1_t1') from filename '1_t1.mha'
        key = f.replace('.mha', '')
        image_files_map[key] = os.path.join(image_dir, f)

for f in os.listdir(mask_dir):
    if f.endswith('.mha'):
        key = f.replace('.mha', '')
        mask_files_map[key] = os.path.join(mask_dir, f)

print(f"\nFound {len(image_files_map)} potential image files and {len(mask_files_map)} potential mask files.")


In [ ]:
# Find common keys (subject/scan IDs) that have both an image and a mask
common_keys = sorted(list(set(image_files_map.keys()) & set(mask_files_map.keys())))
len(common_keys)

In [ ]:
t1_count = 0
t2_count = 0
t2_SPACE_count = 0

patient_t1_scans = set()
patient_t2_scans = set()
patient_t2_SPACE_scans = set()

for key in image_files_map.keys():
    # Extract patient ID (numeric part before '_')
    patient_id = key.split('_')[0]

    if '_t1' in key.lower():
        t1_count += 1
        patient_t1_scans.add(patient_id)
    elif '_t2_space' in key.lower():
        t2_SPACE_count += 1
        patient_t2_SPACE_scans.add(patient_id)
    elif '_t2' in key.lower(): # This will catch '_t2' but not '_t2_SPACE' as it's handled above
        t2_count += 1
        patient_t2_scans.add(patient_id)

# Find patients who have both t1 and (t2 or t2_SPACE) scans
patients_with_any_t2 = patient_t2_scans.union(patient_t2_SPACE_scans)
patients_with_both_t1_and_any_t2 = patient_t1_scans.intersection(patients_with_any_t2)
patients_with_both_t1_and_only_t2 = patient_t1_scans.intersection(patient_t2_scans)

print(f"Total t1 images found: {t1_count}")
print(f"Total t2 images (excluding t2_SPACE) found: {t2_count}")
print(f"Total t2_SPACE images found: {t2_SPACE_count}")
print(f"Number of patients with both t1 and (t2 or t2_SPACE) scans: {len(patients_with_both_t1_and_any_t2)}")
print(f"Number of patients with both t1 and t2(not t2_SPACE) scans: {len(patients_with_both_t1_and_only_t2)}")

In [ ]:
def display_image_mask_slices(example_key):
    """
    Displays each slice of an image and its mask separately with correct physical aspect ratio.
    Iterates slice by slice to ensure appropriate scaling and readability.
    """
    example_image_path = image_files_map[example_key]
    example_mask_path = mask_files_map[example_key]

    # Load images
    sitk_image = sitk.ReadImage(example_image_path)
    sitk_mask = sitk.ReadImage(example_mask_path)

    # Convert to NumPy: (Z, Y, X)
    image_np = sitk.GetArrayFromImage(sitk_image)
    mask_np = sitk.GetArrayFromImage(sitk_mask)

    # Get Spacing: (SpacingX, SpacingY, SpacingZ)
    # We are displaying slices along the X-axis (image_np index 2)
    # Each slice has dimensions: Height=Z, Width=Y
    spacing = sitk_image.GetSpacing()
    spacing_y = spacing[1]
    spacing_z = spacing[2]

    # Calculate physical aspect ratio (Width / Height)
    # If pixels are not square, this ensures they are displayed correctly
    # aspect = spacing_y / spacing_z

    num_slices = image_np.shape[2]
    print(f"\nPatient: {example_key} | Total Slices: {num_slices}")
    print(f"Image Shape (Z, Y, X): {image_np.shape}")

    # Determine figsize based on image size to maintain proportions
    # We set a base height (e.g., 6 inches) and calculate width accordingly
    base_height = 6
    pixel_h, pixel_w = image_np.shape[0], image_np.shape[1]
    physical_w = pixel_w * spacing_y
    physical_h = pixel_h * spacing_z

    # Corrected width for a single image subplot based on aspect
    calc_width = base_height * (physical_w / physical_h)

    for i in range(num_slices):
        # Create a new figure for every slice: [Original, Mask Overlay]
        fig, axes = plt.subplots(1, 2, figsize=(max(10, calc_width * 2), base_height))

        slice_img = image_np[:, :, i]
        slice_mask = mask_np[:, :, i]

        # 1. Original Image
        axes[0].imshow(slice_img, cmap='gray', aspect='equal')
        axes[0].set_title(f"Slice {i} - Original")
        axes[0].axis('off')

        # 2. Mask Overlay
        axes[1].imshow(slice_img, cmap='gray', aspect='equal')
        masked_data = np.ma.masked_where(slice_mask == 0, slice_mask)
        axes[1].imshow(masked_data, cmap='viridis', alpha=0.5, aspect='equal')
        axes[1].set_title(f"Slice {i} - Mask Overlay")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
display_image_mask_slices('100_t1')

In [ ]:
display_image_mask_slices('100_t2')

- 전체 21장 중 끝부분은 척추가 거의 보이지 않아 쓸모 없음.
- 중앙 부분문 골라서 쓰는 것이 좋아보임

# 3. EDA (탐색적 데이터 분석)

In [ ]:
# Exclude the 'Patient' column (index 0)
columns_to_plot = df_gradings.columns[1:]

# Determine the number of rows and columns for subplots
num_columns = len(columns_to_plot)
num_cols_per_row = 3 # You can adjust this for better layout
num_rows = (num_columns + num_cols_per_row - 1) // num_cols_per_row

plt.figure(figsize=(num_cols_per_row * 6, num_rows * 5))
plt.suptitle('Distribution of Target Variables (Class Imbalance Check)', fontsize=16, y=1.02)

for i, col in enumerate(columns_to_plot):
    plt.subplot(num_rows, num_cols_per_row, i + 1)

    # Get value counts for the current column
    value_counts = df_gradings[col].value_counts()

    # Plot as a bar chart
    # Explicitly convert index to string to ensure categorical treatment
    # Removed redundant 'hue' parameter
    sns.barplot(x=value_counts.index.astype(str), y=value_counts.values, palette='Reds')
    plt.title(f'Column: {col}')
    plt.xlabel('Value')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')

    # Add count labels on top of bars
    for index, value in enumerate(value_counts.values):
        plt.text(index, value + (max(value_counts.values) * 0.02), str(value), ha='center', va='bottom')

plt.tight_layout()
plt.show()

- 1~6만 쓰면 됨 (S1 ~ T1)

In [ ]:
# Exclude the 'Patient' column (index 0)
df_for_corr = df_gradings.iloc[:, 1:]

# Calculate the correlation matrix
correlation_matrix = df_for_corr.corr()

# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Heatmap of Grading Variables (Excluding Patient ID)')
plt.show()

In [ ]:
# 파일 최대 값
max(list(map(lambda x: int(x.split('_')[0]), list(image_files_map.keys()))))

In [ ]:
df_gradings['Patient'].value_counts()

In [ ]:
df_gradings['Patient'].value_counts().value_counts().sort_index()

### 환자 정보 통합

In [ ]:
# 1. Get unique patient IDs from df_gradings
unique_patients_gradings = df_gradings['Patient'].unique()

# 2. Determine image presence for each patient (re-compute for clarity and independence)
patient_t1_scans = set()
patient_t2_scans = set() # Excludes T2_SPACE
patient_t2_SPACE_scans = set()

for key in image_files_map.keys():
    patient_id = int(key.split('_')[0]) # Convert patient ID part of key to int

    if '_t1' in key.lower():
        patient_t1_scans.add(patient_id)
    elif '_t2_space' in key.lower():
        patient_t2_SPACE_scans.add(patient_id)
    elif '_t2' in key.lower() and '_t2_space' not in key.lower(): # Explicitly check for T2, excluding T2_SPACE
        patient_t2_scans.add(patient_id)

# 3. Get the number of slices for T2 images for each patient
t2_slice_counts = {}
for patient_id_int in patient_t2_scans:
    patient_id_str = str(patient_id_int)
    t2_key = None
    for key in image_files_map.keys():
        if key.startswith(patient_id_str + '_') and '_t2' in key.lower() and '_t2_space' not in key.lower():
            t2_key = key
            break

    if t2_key:
        image_path = image_files_map[t2_key]
        try:
            sitk_image = sitk.ReadImage(image_path)
            t2_slice_counts[patient_id_int] = sitk_image.GetSize()[0] # Get Z-dimension size
        except Exception as e:
            print(f"Error reading T2 image for patient {patient_id_int} ({t2_key}): {e}")
            t2_slice_counts[patient_id_int] = np.nan # Use NaN if unable to read

# 4. Get the count of gradings for each patient
gradings_count_series = df_gradings['Patient'].value_counts()

# 5. Assemble the combined_df
combined_data = []
for patient_id in sorted(unique_patients_gradings): # Sort for consistent output
    row = {
        'Patient': patient_id,
        't1_present': patient_id in patient_t1_scans,
        't2_present': patient_id in patient_t2_scans,
        't2_space_present': patient_id in patient_t2_SPACE_scans,
        't2_slices': t2_slice_counts.get(patient_id, np.nan), # Get slices, NaN if no T2 or error
        'gradings_count': gradings_count_series.get(patient_id, 0) # Get count, 0 if no gradings
    }
    combined_data.append(row)

combined_df = pd.DataFrame(combined_data)

print("Combined DataFrame created successfully.")


In [ ]:
combined_df.head()

### T2 슬라이스 수 및 방사선 소견 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.histplot(combined_df['t2_slices'].dropna(), kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribution of T2 Slices per Patient')
axes[0].set_xlabel('Number of T2 Slices')
axes[0].set_ylabel('Count')

sns.histplot(combined_df['gradings_count'], kde=True, ax=axes[1], color='lightcoral')
axes[1].set_title('Distribution of Grading Counts per Patient')
axes[1].set_xlabel('Number of Gradings')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
combined_df['t2_slices'].describe()

In [ ]:
patient_min_t2_slices = combined_df.loc[combined_df['t2_slices'].idxmin()]
print(f"Patient with the fewest T2 slices: {patient_min_t2_slices['Patient']} with {patient_min_t2_slices['t2_slices']} slices.")

In [ ]:
display_image_mask_slices('58_t2')

In [ ]:
display_image_mask_slices('203_t2')

# 4. Disc Herniation 분류 실험

양성 비율 ~5.7%의 클래스 불균형 문제. 두 가지 방법을 시도했으나 모두 trivial solution으로 수렴.

## 4-1. 전척추 이미지 다중 레이블 분류

전척추 시상면 이미지를 224×224로 입력, 6개 추간판(IVD 1~6)에 대한 다중 레이블 분류.
3-Conv + FC 모델 (~25.7M params), BCEWithLogitsLoss, Adam lr=1e-4, patience=10.

In [ ]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

class SpiderDataset(Dataset):
    def __init__(self, patient_ids, labels_map, img_files_map, img_size=(224, 224)):
        self.labels_map = labels_map
        self.img_size = img_size
        self.data = [
            (pid, img_files_map[f"{pid}_t2"])
            for pid in patient_ids
            if f"{pid}_t2" in img_files_map
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pid, path = self.data[idx]
        arr = sitk.GetArrayFromImage(sitk.ReadImage(path))  # (Z, Y, X)

        # 가장 짧은 축 = 시상면 슬라이스 방향
        ax = int(np.argmin(arr.shape))
        mid = arr.shape[ax] // 2
        if ax == 0:
            sl = arr[mid, :, :]
        elif ax == 1:
            sl = arr[:, mid, :]
        else:
            sl = arr[:, :, mid]

        img = torch.from_numpy(sl.astype(np.float32)).unsqueeze(0)  # (1, H, W)
        img = T.Resize(self.img_size)(img)
        label = torch.tensor(self.labels_map[pid], dtype=torch.float32)
        return img, label


# ── 레이블 구성 (IVD 1~6 모두 있는 환자만) ─────────────────────────────────
target_ivds = list(range(1, 7))
df_ivd = df_gradings[df_gradings['IVD label'].isin(target_ivds)]

patient_labels_map = {}
for pid in df_ivd['Patient'].unique():
    rows = df_ivd[df_ivd['Patient'] == pid].sort_values('IVD label')
    if len(rows) == len(target_ivds):
        patient_labels_map[str(pid)] = rows['Disc herniation'].tolist()

print(f"사용 환자 수: {len(patient_labels_map)}")
pos = sum(sum(v) for v in patient_labels_map.values())
total = len(patient_labels_map) * 6
print(f"양성 비율: {pos}/{total} ({pos/total*100:.1f}%)")

# ── Train / Val / Test split (70 / 15 / 15) ───────────────────────────────
all_ids = list(patient_labels_map.keys())
train_ids, tmp_ids = train_test_split(all_ids, test_size=0.3, random_state=42)
val_ids, test_ids  = train_test_split(tmp_ids, test_size=0.5, random_state=42)
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

train_loader = DataLoader(SpiderDataset(train_ids, patient_labels_map, image_files_map),
                          batch_size=16, shuffle=True)
val_loader   = DataLoader(SpiderDataset(val_ids,   patient_labels_map, image_files_map),
                          batch_size=16, shuffle=False)
test_loader  = DataLoader(SpiderDataset(test_ids,  patient_labels_map, image_files_map),
                          batch_size=16, shuffle=False)


In [ ]:
import torch.nn as nn

class SpiderCNN(nn.Module):
    def __init__(self, num_discs=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        # 224 → 112 → 56 → 28  =>  128 * 28 * 28
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_discs),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = SpiderCNN().to(device)
print(model)
print(f"\nDevice: {device}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 파라미터 수: {total_params:,}")


In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt
import datetime, os

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

max_epochs = 50
patience = 10
best_val_loss = float('inf')
wait = 0
train_losses, val_losses = [], []

print("학습 시작...")
for epoch in range(max_epochs):
    # ── Train ──
    model.train()
    t_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    # ── Validation ──
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            v_loss += criterion(model(imgs), labels).item()
    v_loss /= len(val_loader)

    train_losses.append(t_loss)
    val_losses.append(v_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}: train={t_loss:.4f}  val={v_loss:.4f}")

    # ── Early Stopping ──
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        wait = 0
        torch.save(model.state_dict(), 'best_baseline.pth')
    else:
        wait += 1
        if wait >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

# best weight 복원 후 저장
model.load_state_dict(torch.load('best_baseline.pth', map_location=device))
os.makedirs('./models', exist_ok=True)
now = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = f'./models/baseline_{now}.pth'
torch.save(model.state_dict(), save_path)
print(f"\n모델 저장: {save_path}")

# ── Loss curve ──
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Baseline Training Curve')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

DISC_NAMES = [f'IVD {i}' for i in range(1, 7)]

def evaluate(loader, split_name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = model(imgs.to(device))
            preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    preds  = np.vstack(all_preds)   # (N, 6)
    labels = np.vstack(all_labels)  # (N, 6)

    # ── 지표 테이블 ──────────────────────────────────────────────────────
    rows = []
    for i, name in enumerate(DISC_NAMES):
        rows.append({
            'Disc':      name,
            'Accuracy':  accuracy_score(labels[:, i], preds[:, i]),
            'Precision': precision_score(labels[:, i], preds[:, i], zero_division=0),
            'Recall':    recall_score(labels[:, i], preds[:, i], zero_division=0),
            'F1':        f1_score(labels[:, i], preds[:, i], zero_division=0),
        })
    # macro avg
    rows.append({
        'Disc':      'Macro Avg',
        'Accuracy':  np.mean([r['Accuracy']  for r in rows]),
        'Precision': np.mean([r['Precision'] for r in rows]),
        'Recall':    np.mean([r['Recall']    for r in rows]),
        'F1':        np.mean([r['F1']        for r in rows]),
    })
    df_result = pd.DataFrame(rows).set_index('Disc').round(4)
    print(f"\n{'='*50}")
    print(f"{split_name} 평가 결과")
    print('='*50)
    print(df_result.to_string())

    # ── Confusion Matrix (2행 3열) ───────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    fig.suptitle(f'Confusion Matrices — {split_name}', fontsize=14)
    for i, (name, ax) in enumerate(zip(DISC_NAMES, axes.ravel())):
        cm = confusion_matrix(labels[:, i], preds[:, i], labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Normal', 'Herniation'],
                    yticklabels=['Normal', 'Herniation'])
        ax.set_title(name)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

evaluate(val_loader,  "Validation Set")
evaluate(test_loader, "Test Set")


## 4-2. 디스크 크롭 + Focal Loss

마스크를 활용해 추간판(IVD)별 바운딩 박스로 크롭 후 128×128로 리사이즈.
Focal Loss(α=0.25, γ=2.0)로 클래스 불균형 보완, patience=10.

In [ ]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

class SpiderDiscDataset(Dataset):
    """추간판 1개 = 샘플 1개. 마스크로 크롭 후 이진분류."""

    def __init__(self, samples, labels, img_files_map, mask_files_map, img_size=(128, 128)):
        # samples: list of (pid_str, disc_idx)  disc_idx in 1..6
        # labels:  dict (pid_str, disc_idx) -> 0 or 1
        self.samples = samples
        self.labels = labels
        self.img_files_map = img_files_map
        self.mask_files_map = mask_files_map
        self.img_size = img_size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, disc_idx = self.samples[idx]
        img_key  = f"{pid}_t2"
        mask_key = f"{pid}_t2"

        sitk_img  = sitk.ReadImage(self.img_files_map[img_key])
        sitk_mask = sitk.ReadImage(self.mask_files_map[mask_key])
        img_arr   = sitk.GetArrayFromImage(sitk_img)   # (Z, Y, X)
        mask_arr  = sitk.GetArrayFromImage(sitk_mask)
        spacing   = sitk_img.GetSpacing()              # (sx, sy, sz) mm/pixel

        # 최단 축 = 시상면 슬라이스 방향 (베이스라인과 동일)
        ax  = int(np.argmin(img_arr.shape))
        mid = img_arr.shape[ax] // 2

        if ax == 0:
            img_2d, mask_2d = img_arr[mid, :, :], mask_arr[mid, :, :]
            sh, sw = spacing[1], spacing[0]
        elif ax == 1:
            img_2d, mask_2d = img_arr[:, mid, :], mask_arr[:, mid, :]
            sh, sw = spacing[2], spacing[0]
        else:
            img_2d, mask_2d = img_arr[:, :, mid], mask_arr[:, :, mid]
            sh, sw = spacing[2], spacing[1]

        # PixelSpacing 기반 40mm 물리적 여백
        margin_h = int(40.0 / sh) if sh > 0 else 20
        margin_w = int(40.0 / sw) if sw > 0 else 20

        # 디스크 bounding box (mask 201~206)
        disc_mask = (mask_2d == (200 + disc_idx))
        rows = np.where(disc_mask.any(axis=1))[0]
        cols = np.where(disc_mask.any(axis=0))[0]

        if len(rows) == 0:
            # 중앙 슬라이스에 해당 디스크 없음 → 전체 슬라이스 fallback
            crop = img_2d
        else:
            rmin = max(0, rows[0]  - margin_h)
            rmax = min(img_2d.shape[0] - 1, rows[-1] + margin_h)
            cmin = max(0, cols[0]  - margin_w)
            cmax = min(img_2d.shape[1] - 1, cols[-1] + margin_w)
            crop = img_2d[rmin:rmax+1, cmin:cmax+1]

        img_tensor = T.Resize(self.img_size)(
            torch.from_numpy(crop.astype(np.float32)).unsqueeze(0)
        )
        label = torch.tensor([self.labels[(pid, disc_idx)]], dtype=torch.float32)
        return img_tensor, label


# ── 레이블 구성 ───────────────────────────────────────────────────────────
target_ivds = list(range(1, 7))
df_ivd = df_gradings[df_gradings['IVD label'].isin(target_ivds)]

disc_labels = {}   # (pid_str, disc_idx) -> 0/1
valid_pids  = set()

for pid in df_ivd['Patient'].unique():
    rows = df_ivd[df_ivd['Patient'] == pid].sort_values('IVD label')
    if len(rows) == len(target_ivds):
        pid_str = str(pid)
        if f"{pid_str}_t2" not in image_files_map:
            continue
        for _, row in rows.iterrows():
            disc_labels[(pid_str, int(row['IVD label']))] = int(row['Disc herniation'])
        valid_pids.add(pid_str)

all_samples = [(pid, d) for pid in valid_pids for d in range(1, 7)]
pos = sum(disc_labels[s] for s in all_samples)
print(f"환자: {len(valid_pids)}, 샘플: {len(all_samples)}, 양성: {pos} ({pos/len(all_samples)*100:.1f}%)")

# ── Patient-level split (70/15/15, 누수 방지) ────────────────────────────
pid_list = list(valid_pids)
train_pids, tmp_pids = train_test_split(pid_list, test_size=0.3, random_state=42)
val_pids, test_pids  = train_test_split(tmp_pids, test_size=0.5, random_state=42)

train_pids_set, val_pids_set, test_pids_set = set(train_pids), set(val_pids), set(test_pids)
train_samples = [(p, d) for p, d in all_samples if p in train_pids_set]
val_samples   = [(p, d) for p, d in all_samples if p in val_pids_set]
test_samples  = [(p, d) for p, d in all_samples if p in test_pids_set]
print(f"Split — Train:{len(train_samples)}, Val:{len(val_samples)}, Test:{len(test_samples)}")

train_loader_disc = DataLoader(
    SpiderDiscDataset(train_samples, disc_labels, image_files_map, mask_files_map),
    batch_size=32, shuffle=True)
val_loader_disc = DataLoader(
    SpiderDiscDataset(val_samples, disc_labels, image_files_map, mask_files_map),
    batch_size=32, shuffle=False)
test_loader_disc = DataLoader(
    SpiderDiscDataset(test_samples, disc_labels, image_files_map, mask_files_map),
    batch_size=32, shuffle=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk

def get_disc_crop_with_mask(pid, disc_idx):
    """(크롭 이미지, 디스크 마스크 2D) 반환"""
    sitk_img  = sitk.ReadImage(image_files_map[f"{pid}_t2"])
    sitk_mask = sitk.ReadImage(mask_files_map[f"{pid}_t2"])
    img_arr   = sitk.GetArrayFromImage(sitk_img)
    mask_arr  = sitk.GetArrayFromImage(sitk_mask)
    spacing   = sitk_img.GetSpacing()

    ax  = int(np.argmin(img_arr.shape))
    mid = img_arr.shape[ax] // 2

    if ax == 0:
        img_2d, mask_2d = img_arr[mid, :, :], mask_arr[mid, :, :]
        sh, sw = spacing[1], spacing[0]
    elif ax == 1:
        img_2d, mask_2d = img_arr[:, mid, :], mask_arr[:, mid, :]
        sh, sw = spacing[2], spacing[0]
    else:
        img_2d, mask_2d = img_arr[:, :, mid], mask_arr[:, :, mid]
        sh, sw = spacing[2], spacing[1]

    margin_h = int(40.0 / sh) if sh > 0 else 20
    margin_w = int(40.0 / sw) if sw > 0 else 20

    disc_mask = (mask_2d == (200 + disc_idx))
    rows = np.where(disc_mask.any(axis=1))[0]
    cols = np.where(disc_mask.any(axis=0))[0]

    if len(rows) == 0:
        return None, None

    rmin = max(0, rows[0]  - margin_h)
    rmax = min(img_2d.shape[0]-1, rows[-1] + margin_h)
    cmin = max(0, cols[0]  - margin_w)
    cmax = min(img_2d.shape[1]-1, cols[-1] + margin_w)

    crop_img  = img_2d[rmin:rmax+1, cmin:cmax+1].astype(np.float32)
    crop_mask = disc_mask[rmin:rmax+1, cmin:cmax+1]
    return crop_img, crop_mask


# 양성/음성 각 6개씩 예시 선택
pos_examples = [(pid, d) for (pid, d), v in disc_labels.items() if v == 1][:6]
neg_examples = [(pid, d) for (pid, d), v in disc_labels.items() if v == 0][:6]
examples = pos_examples + neg_examples

n_cols = 6
n_rows = 2  # 양성 행 / 음성 행
fig, axes = plt.subplots(n_rows * 2, n_cols, figsize=(n_cols * 3, n_rows * 2 * 3))
fig.suptitle("Disc Crop 예시  (위: Herniation=1, 아래: Herniation=0)\n"
             "홀수 행: 원본 크롭 | 짝수 행: 마스크 오버레이", fontsize=12)

for col, (pid, disc_idx) in enumerate(examples):
    row_base = 0 if col < n_cols else 2   # 양성=행0,1 / 음성=행2,3
    col_idx  = col % n_cols
    label_val = disc_labels[(pid, disc_idx)]

    crop_img, crop_mask = get_disc_crop_with_mask(pid, disc_idx)
    if crop_img is None:
        continue

    # 원본 크롭
    ax_img = axes[row_base][col_idx]
    ax_img.imshow(crop_img, cmap='gray')
    ax_img.set_title(f"P{pid} IVD{disc_idx}\nHern={label_val}", fontsize=8)
    ax_img.axis('off')

    # 마스크 오버레이
    ax_ov = axes[row_base + 1][col_idx]
    ax_ov.imshow(crop_img, cmap='gray')
    masked = np.ma.masked_where(~crop_mask, crop_mask.astype(float))
    ax_ov.imshow(masked, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
    ax_ov.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce  = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()


class SpiderDiscCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        # 128 → 64 → 32 → 16  =>  128 * 16 * 16 = 32768
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


device    = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_imp = SpiderDiscCNN().to(device)
print(model_imp)
total = sum(p.numel() for p in model_imp.parameters() if p.requires_grad)
print(f"\nDevice: {device} | 학습 파라미터: {total:,}")


In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt
import datetime, os

criterion_imp = FocalLoss(alpha=0.25, gamma=2.0)
optimizer_imp = optim.Adam(model_imp.parameters(), lr=1e-4)

max_epochs = 50
patience   = 10
best_val_loss = float('inf')
wait = 0
train_losses_imp, val_losses_imp = [], []

print("학습 시작 (Disc Crop + Focal Loss)...")
for epoch in range(max_epochs):
    model_imp.train()
    t_loss = 0.0
    for imgs, labels in train_loader_disc:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_imp.zero_grad()
        loss = criterion_imp(model_imp(imgs), labels)
        loss.backward()
        optimizer_imp.step()
        t_loss += loss.item()
    t_loss /= len(train_loader_disc)

    model_imp.eval()
    v_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader_disc:
            imgs, labels = imgs.to(device), labels.to(device)
            v_loss += criterion_imp(model_imp(imgs), labels).item()
    v_loss /= len(val_loader_disc)

    train_losses_imp.append(t_loss)
    val_losses_imp.append(v_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}: train={t_loss:.4f}  val={v_loss:.4f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        wait = 0
        torch.save(model_imp.state_dict(), 'best_improved.pth')
    else:
        wait += 1
        if wait >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

model_imp.load_state_dict(torch.load('best_improved.pth', map_location=device))
os.makedirs('./models', exist_ok=True)
now = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = f'./models/improved_{now}.pth'
torch.save(model_imp.state_dict(), save_path)
print(f"\n모델 저장: {save_path}")

plt.figure(figsize=(8, 4))
plt.plot(train_losses_imp, label='Train Loss')
plt.plot(val_losses_imp,   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Focal Loss')
plt.title('Improved Model Training Curve (Disc Crop + Focal Loss)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

def evaluate_disc(loader, split_name):
    model_imp.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = model_imp(imgs.to(device))
            preds  = (torch.sigmoid(logits) > 0.5).cpu().numpy().flatten()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy().flatten())

    preds  = np.array(all_preds)
    labels = np.array(all_labels)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)

    print(f"\n{'='*45}")
    print(f"{split_name}")
    print(f"{'='*45}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1 Score : {f1:.4f}")

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Herniation'],
                yticklabels=['Normal', 'Herniation'])
    plt.title(f'Confusion Matrix — {split_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

evaluate_disc(val_loader_disc,  "Validation Set")
evaluate_disc(test_loader_disc, "Test Set")


## 4-3. GAP 경량 모델 + Focal Loss (Stratified Split, 200 epoch)

2-Conv + Global Average Pooling 경량 모델 (~19K params), 환자 단위 계층화 분할.
Sec 4-2와 동일한 Focal Loss 유지.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Sec 5-1: SpiderDiscCNN_GAP + FocalLoss + Stratified Split  seed=0
#   사용 변수: disc_labels, valid_pids, all_samples, image_files_map,
#             mask_files_map, SpiderDiscDataset, FocalLoss, device
# ─────────────────────────────────────────────────────────────────────
import numpy as np, torch, torch.nn as nn, torch.optim as optim, random
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

SEED_S1       = 0
MAX_EPOCHS_S1 = 200
PATIENCE_S1   = 15
LOG_EVERY     = 10   # 몇 epoch마다 출력할지

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# ── GAP 모델 정의 ─────────────────────────────────────────────────────
class SpiderDiscCNN_GAP_51(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.head(self.gap(self.features(x)))

# ── 계층화 분할 ───────────────────────────────────────────────────────
def make_split_stratified_s1(seed):
    pids = sorted(valid_pids)
    strat = [int(any(disc_labels.get((pid, d), 0) == 1 for d in range(1, 7)))
             for pid in pids]
    pids_trn, pids_rest, _, y_rest = train_test_split(
        pids, strat, test_size=0.30, stratify=strat, random_state=seed)
    pids_val, pids_tst = train_test_split(
        pids_rest, test_size=0.50, stratify=y_rest, random_state=seed)
    trn = [(p, d) for p, d in all_samples if p in set(pids_trn)]
    val = [(p, d) for p, d in all_samples if p in set(pids_val)]
    tst = [(p, d) for p, d in all_samples if p in set(pids_tst)]
    return trn, val, tst

def pos_rate_s1(samples):
    labels = [disc_labels[k] for k in samples]
    return sum(labels) / len(labels)

# ── Split 확인 ────────────────────────────────────────────────────────
set_seed(SEED_S1)
trn_s1, val_s1, tst_s1 = make_split_stratified_s1(SEED_S1)
print(f'Split  → train:{len(trn_s1)}  val:{len(val_s1)}  test:{len(tst_s1)}')
print(f'PosRate→ train:{pos_rate_s1(trn_s1):.3f}  val:{pos_rate_s1(val_s1):.3f}  test:{pos_rate_s1(tst_s1):.3f}')
print()

trn_ld_s1 = DataLoader(SpiderDiscDataset(trn_s1, disc_labels, image_files_map, mask_files_map),
                       batch_size=32, shuffle=True)
val_ld_s1 = DataLoader(SpiderDiscDataset(val_s1, disc_labels, image_files_map, mask_files_map),
                       batch_size=32, shuffle=False)
tst_ld_s1 = DataLoader(SpiderDiscDataset(tst_s1, disc_labels, image_files_map, mask_files_map),
                       batch_size=32, shuffle=False)

# ── 학습 ──────────────────────────────────────────────────────────────
model_s1 = SpiderDiscCNN_GAP_51().to(device)
crit_s1  = FocalLoss(alpha=0.25, gamma=2.0)
opt_s1   = optim.Adam(model_s1.parameters(), lr=1e-4)
total_p  = sum(p.numel() for p in model_s1.parameters() if p.requires_grad)
print(f'파라미터: {total_p:,}  |  Device: {device}')
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val F1':>7} | {'Wait':>4}")
print('-' * 52)

best_val_loss, wait, best_state = float('inf'), 0, None
stopped_epoch = MAX_EPOCHS_S1

for epoch in range(MAX_EPOCHS_S1):
    # Train
    model_s1.train()
    t_loss = 0.0
    for imgs, lbls in trn_ld_s1:
        opt_s1.zero_grad()
        loss = crit_s1(model_s1(imgs.to(device)), lbls.to(device))
        loss.backward()
        opt_s1.step()
        t_loss += loss.item()
    t_loss /= len(trn_ld_s1)

    # Val loss
    model_s1.eval()
    v_loss = 0.0
    with torch.no_grad():
        for imgs, lbls in val_ld_s1:
            v_loss += crit_s1(model_s1(imgs.to(device)), lbls.to(device)).item()
    v_loss /= len(val_ld_s1)

    # Early stopping
    if v_loss < best_val_loss:
        best_val_loss, wait = v_loss, 0
        best_state = {k: v.clone() for k, v in model_s1.state_dict().items()}
    else:
        wait += 1

    # 중간 출력 (LOG_EVERY 마다 + 마지막 epoch)
    if (epoch + 1) % LOG_EVERY == 0 or epoch == 0 or wait >= PATIENCE_S1:
        model_s1.eval()
        preds, labs = [], []
        with torch.no_grad():
            for imgs, lbls in val_ld_s1:
                p = (torch.sigmoid(model_s1(imgs.to(device))) > 0.5).cpu().numpy().flatten()
                preds.extend(p); labs.extend(lbls.numpy().flatten())
        vf1_mid = f1_score(labs, preds, zero_division=0)
        print(f"{epoch+1:>6} | {t_loss:>10.4f} | {v_loss:>9.4f} | {vf1_mid:>7.4f} | {wait:>4}")

    if wait >= PATIENCE_S1:
        stopped_epoch = epoch + 1
        print(f'Early stop @ epoch {stopped_epoch}')
        break

model_s1.load_state_dict(best_state)

# ── 최종 평가 ─────────────────────────────────────────────────────────
def get_f1_s1(loader):
    model_s1.eval(); preds, labs = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            p = (torch.sigmoid(model_s1(imgs.to(device))) > 0.5).cpu().numpy().flatten()
            preds.extend(p); labs.extend(lbls.numpy().flatten())
    return f1_score(labs, preds, zero_division=0)

print()
vf1 = get_f1_s1(val_ld_s1)
tf1 = get_f1_s1(tst_ld_s1)
print(f'Val  F1 : {vf1:.4f}')
print(f'Test F1 : {tf1:.4f}')
print(f'Gap     : {abs(vf1-tf1):.4f}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

def evaluate_s1(loader, split_name):
    model_s1.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = model_s1(imgs.to(device))
            preds  = (torch.sigmoid(logits) > 0.5).cpu().numpy().flatten()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy().flatten())

    preds  = np.array(all_preds)
    labels = np.array(all_labels)

    print(f"\n{'='*45}")
    print(f"{split_name} (Sec 5-1)")
    print(f"{'='*45}")
    print(f"  Accuracy : {accuracy_score(labels, preds):.4f}")
    print(f"  Precision: {precision_score(labels, preds, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(labels, preds, zero_division=0):.4f}")
    print(f"  F1 Score : {f1_score(labels, preds, zero_division=0):.4f}")

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Herniation'],
                yticklabels=['Normal', 'Herniation'])
    plt.title(f'Confusion Matrix — {split_name} (Sec 5-1)')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout(); plt.show()

evaluate_s1(val_ld_s1,  'Validation Set')
evaluate_s1(tst_ld_s1,  'Test Set')


# 5. Disc Bulging 분류 베이스라인

Herniation 실패(양성 ~5.7%) 후 Disc Bulging(양성 ~49%)으로 타겟 변경.
균형 잡힌 클래스 덕분에 BCE 손실만으로 학습 가능.

In [ ]:
# ── Disc Bulging 레이블 구성 ─────────────────────────────────────
target_ivds = list(range(1, 7))
df_ivd = df_gradings[df_gradings['IVD label'].isin(target_ivds)]

bulging_labels = {}   # (pid_str, disc_idx) -> 0/1
valid_pids_b   = set()

for pid in df_ivd['Patient'].unique():
    rows = df_ivd[df_ivd['Patient'] == pid].sort_values('IVD label')
    if len(rows) == len(target_ivds):
        pid_str = str(pid)
        if f"{pid_str}_t2" not in image_files_map:
            continue
        for _, row in rows.iterrows():
            bulging_labels[(pid_str, int(row['IVD label']))] = int(row['Disc bulging'])
        valid_pids_b.add(pid_str)

all_samples_b = [(pid, d) for pid in valid_pids_b for d in range(1, 7)]
pos_b = sum(bulging_labels[s] for s in all_samples_b)
print(f"환자: {len(valid_pids_b)}, 샘플: {len(all_samples_b)}")
print(f"양성(Bulging): {pos_b} / {len(all_samples_b)} ({pos_b/len(all_samples_b)*100:.1f}%)")

## 5-1. 단일 Hold-out 실험 (SpiderDiscCNN_GAP, seed=0)

GAP 경량 모델로 단일 hold-out(70/15/15) 실험.
Val F1=0.808이지만 Test F1=0.698 → Val-Test 갭 0.111. 평가 신뢰도 문제 확인.

In [ ]:
import random, numpy as np, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt, datetime, os
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# ── 재현성 ───────────────────────────────────────────────────────────
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Split (seed=0) ────────────────────────────────────────────────────
pid_list_b0 = list(valid_pids_b)
train_pids_b0, tmp_b0 = train_test_split(pid_list_b0, test_size=0.3, random_state=SEED)
val_pids_b0,  test_pids_b0 = train_test_split(tmp_b0,  test_size=0.5, random_state=SEED)

train_s_b0 = [(p, d) for p, d in all_samples_b if p in set(train_pids_b0)]
val_s_b0   = [(p, d) for p, d in all_samples_b if p in set(val_pids_b0)]
test_s_b0  = [(p, d) for p, d in all_samples_b if p in set(test_pids_b0)]
print(f'Split — Train:{len(train_s_b0)}, Val:{len(val_s_b0)}, Test:{len(test_s_b0)}')

train_loader_b0 = DataLoader(
    SpiderDiscDataset(train_s_b0, bulging_labels, image_files_map, mask_files_map),
    batch_size=32, shuffle=True)
val_loader_b0 = DataLoader(
    SpiderDiscDataset(val_s_b0, bulging_labels, image_files_map, mask_files_map),
    batch_size=32, shuffle=False)
test_loader_b0 = DataLoader(
    SpiderDiscDataset(test_s_b0, bulging_labels, image_files_map, mask_files_map),
    batch_size=32, shuffle=False)

# ── 모델 ─────────────────────────────────────────────────────────────
class SpiderDiscCNN_GAP_b0(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, x):
        return self.head(self.gap(self.features(x)))

device   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model_b0 = SpiderDiscCNN_GAP_b0().to(device)
crit_b0  = nn.BCEWithLogitsLoss()
opt_b0   = optim.Adam(model_b0.parameters(), lr=1e-4)
total_p  = sum(p.numel() for p in model_b0.parameters() if p.requires_grad)
print(f'파라미터: {total_p:,}  |  Device: {device}')

# ── 학습 ─────────────────────────────────────────────────────────────
LOG_EVERY = 10
max_epochs, patience = 200, 10
best_val, wait, best_state = float('inf'), 0, None
print(f"{'Epoch':>6} | {'Train':>8} | {'Val':>8} | {'Val F1':>7} | {'Wait':>4}")
print('-' * 48)

from sklearn.metrics import f1_score

for epoch in range(max_epochs):
    model_b0.train()
    t_loss = 0.0
    for imgs, lbls in train_loader_b0:
        imgs, lbls = imgs.to(device), lbls.to(device)
        opt_b0.zero_grad()
        loss = crit_b0(model_b0(imgs), lbls)
        loss.backward(); opt_b0.step()
        t_loss += loss.item()
    t_loss /= len(train_loader_b0)

    model_b0.eval()
    v_loss, preds, labs = 0.0, [], []
    with torch.no_grad():
        for imgs, lbls in val_loader_b0:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model_b0(imgs)
            v_loss += crit_b0(logits, lbls).item()
            preds.extend((torch.sigmoid(logits) > 0.5).cpu().numpy().flatten())
            labs.extend(lbls.cpu().numpy().flatten())
    v_loss /= len(val_loader_b0)

    if v_loss < best_val:
        best_val, wait = v_loss, 0
        best_state = {k: v.clone() for k, v in model_b0.state_dict().items()}
    else:
        wait += 1

    if (epoch + 1) % LOG_EVERY == 0 or epoch == 0 or wait >= patience:
        vf1 = f1_score(labs, preds, zero_division=0)
        print(f"{epoch+1:>6} | {t_loss:>8.4f} | {v_loss:>8.4f} | {vf1:>7.4f} | {wait:>4}")

    if wait >= patience:
        print(f'Early stop @ epoch {epoch+1}')
        break

model_b0.load_state_dict(best_state)
os.makedirs('./models', exist_ok=True)
now = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
torch.save(model_b0.state_dict(), f'./models/bulging_b0_{now}.pth')
print(f'모델 저장: ./models/bulging_b0_{now}.pth')


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

def evaluate_b0(loader, split_name):
    model_b0.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = model_b0(imgs.to(device))
            preds  = (torch.sigmoid(logits) > 0.5).cpu().numpy().flatten()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy().flatten())

    preds  = np.array(all_preds)
    labels = np.array(all_labels)

    print(f"\n{'='*45}")
    print(f"{split_name} (Sec 6-1 — GAP Baseline)")
    print(f"{'='*45}")
    print(f"  Accuracy : {accuracy_score(labels, preds):.4f}")
    print(f"  Precision: {precision_score(labels, preds, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(labels, preds, zero_division=0):.4f}")
    print(f"  F1 Score : {f1_score(labels, preds, zero_division=0):.4f}")

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Bulging', 'Bulging'],
                yticklabels=['No Bulging', 'Bulging'])
    plt.title(f'Confusion Matrix — {split_name} (Sec 6-1)')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout(); plt.show()

evaluate_b0(val_loader_b0,  'Validation Set')
evaluate_b0(test_loader_b0, 'Test Set')


## 5-2. 3-Fold Cross-Validation 베이스라인 (Sec 13)

Val-Test 갭 문제 해결을 위해 환자 단위 StratifiedKFold(n=3) 도입.
동일 전처리(bbox crop + Resize 128×128)와 SpiderDiscCNN_GAP 모델 유지.
CV Mean F1 = 0.7613 ± 0.020.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

# ── 결과 저장 폴더 ──────────────────────────────────────────────────────
RESULTS_ROOT = 'results'

def make_exp_dir(exp_name):
    """results/exp_name 폴더 생성 후 경로 반환"""
    path = os.path.join(RESULTS_ROOT, exp_name)
    os.makedirs(path, exist_ok=True)
    return path

# ── 공용 CV 학습 함수 ────────────────────────────────────────────────────
def run_kfold_cv(
    exp_name,
    pid_list, strat_labels, all_samples, labels_dict,
    make_dataset_fn,          # fn(samples) -> Dataset  (val에 사용)
    make_model_fn,            # fn() -> nn.Module
    make_train_dataset_fn=None,  # fn(samples) -> Dataset  (train에만 사용, None이면 make_dataset_fn 사용)
    n_folds=3, max_epochs=50, patience=10,
    batch_size=32, lr=1e-4, seed=0,
    log_every=10,
):
    """
    범용 환자 레벨 Stratified K-Fold CV.
    결과를 results/exp_name/ 폴더에 CSV + 그래프(PDF)로 저장.
    """
    exp_dir = make_exp_dir(exp_name)
    skf     = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    fold_records = []
    fold_curves  = []   # (train_losses, val_losses) per fold
    fold_cms     = []   # confusion matrix per fold

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(pid_list, strat_labels)):
        print(f'\n{"="*55}')
        tr_pids = [pid_list[i] for i in tr_idx]
        va_pids = [pid_list[i] for i in va_idx]
        tr_smp  = [(p, d) for p, d in all_samples if p in set(tr_pids)]
        va_smp  = [(p, d) for p, d in all_samples if p in set(va_pids)]
        print(f'  Fold {fold_idx+1}/{n_folds} | Train {len(tr_pids)} pids ({len(tr_smp)} smp) '
              f'| Val {len(va_pids)} pids ({len(va_smp)} smp)')

        _train_ds_fn = make_train_dataset_fn if make_train_dataset_fn else make_dataset_fn
        tr_loader = DataLoader(_train_ds_fn(tr_smp),
                               batch_size=batch_size, shuffle=True,  num_workers=0)
        va_loader = DataLoader(make_dataset_fn(va_smp),
                               batch_size=batch_size, shuffle=False, num_workers=0)

        model     = make_model_fn().to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        best_val, wait = float('inf'), 0
        best_state = None
        tr_losses, va_losses = [], []

        print(f'  {"Ep":>5} | {"TrLoss":>8} | {"VaLoss":>8} | {"ValF1":>7} | Wait')
        print(f'  {"-"*46}')

        for epoch in range(max_epochs):
            # train
            model.train()
            t = 0.0
            for imgs, lbls in tr_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), lbls)
                loss.backward(); optimizer.step()
                t += loss.item()
            t /= len(tr_loader)

            # val
            model.eval()
            v, preds_ep, lbls_ep = 0.0, [], []
            with torch.no_grad():
                for imgs, lbls in va_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    logits = model(imgs)
                    v += criterion(logits, lbls).item()
                    preds_ep.extend((torch.sigmoid(logits) > 0.5).cpu().numpy().flatten())
                    lbls_ep.extend(lbls.cpu().numpy().flatten())
            v /= len(va_loader)
            vf1 = f1_score(lbls_ep, preds_ep, zero_division=0)

            tr_losses.append(t); va_losses.append(v)

            if (epoch + 1) % log_every == 0 or epoch == 0:
                print(f'  {epoch+1:>5} | {t:>8.4f} | {v:>8.4f} | {vf1:>7.4f} | {wait}')

            if v < best_val:
                best_val, wait = v, 0
                best_state = {k: v2.cpu().clone() for k, v2 in model.state_dict().items()}
            else:
                wait += 1
                if wait >= patience:
                    print(f'  Early stop @ epoch {epoch+1}')
                    break

        model.load_state_dict(best_state)

        # 최종 평가
        model.eval()
        all_p, all_l = [], []
        with torch.no_grad():
            for imgs, lbls in va_loader:
                logits = model(imgs.to(device))
                all_p.extend((torch.sigmoid(logits) > 0.5).cpu().numpy().flatten())
                all_l.extend(lbls.numpy().flatten())
        all_p, all_l = np.array(all_p), np.array(all_l)

        acc  = accuracy_score(all_l, all_p)
        prec = precision_score(all_l, all_p, zero_division=0)
        rec  = recall_score(all_l, all_p, zero_division=0)
        f1   = f1_score(all_l, all_p, zero_division=0)
        cm   = confusion_matrix(all_l, all_p, labels=[0, 1])

        print(f'  → Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}')

        fold_records.append({'fold': fold_idx+1, 'acc': acc,
                             'precision': prec, 'recall': rec, 'f1': f1})
        fold_curves.append((tr_losses, va_losses))
        fold_cms.append(cm)

    # ── Mean / Std 행 추가 ────────────────────────────────────────────
    keys = ['acc', 'precision', 'recall', 'f1']
    mean_row = {'fold': 'mean'}
    std_row  = {'fold': 'std'}
    for k in keys:
        vals = [r[k] for r in fold_records]
        mean_row[k] = round(np.mean(vals), 6)
        std_row[k]  = round(np.std(vals),  6)
    fold_records += [mean_row, std_row]

    # ── CSV 저장 ─────────────────────────────────────────────────────
    df_res = pd.DataFrame(fold_records)
    csv_path = os.path.join(exp_dir, 'metrics.csv')
    df_res.to_csv(csv_path, index=False)
    print(f'\n저장: {csv_path}')
    display(df_res)

    # ── Loss 데이터 .npy 저장 (그래프 재현용) ───────────────────────
    npy_path = os.path.join(exp_dir, 'loss_data.npy')
    np.save(npy_path, {
        'train': [list(tr) for tr, va in fold_curves],
        'val':   [list(va) for tr, va in fold_curves],
    })
    print(f'저장: {npy_path}')

    # ── Loss 데이터 .npy 저장 (그래프 재현용) ───────────────────────
    np.save(os.path.join(exp_dir, 'loss_data.npy'), {
        'train': [list(tr) for tr, va in fold_curves],
        'val':   [list(va) for tr, va in fold_curves],
    })
    print(f'저장: {os.path.join(exp_dir, "loss_data.npy")}')

    # ── Train/Val Loss 곡선 저장 (PDF, 제목 없음, grid 없음) ──────────
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    fig, axes = plt.subplots(1, n_folds, figsize=(4.5*n_folds, 3.5))
    if n_folds == 1:
        axes = [axes]
    for i, (tr, va) in enumerate(fold_curves):
        ax = axes[i]
        ax.plot(tr, label='Train', color=colors[i % len(colors)], alpha=0.85)
        ax.plot(va, label='Val',   color=colors[i % len(colors)], linestyle='--', alpha=0.85)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('BCE Loss')
        ax.annotate(f'Fold {i+1}', xy=(0.05, 0.92), xycoords='axes fraction', fontsize=9)
        ax.legend(fontsize=9)
        ax.grid(False)
    plt.tight_layout()
    loss_path = os.path.join(exp_dir, 'loss_curves.pdf')
    plt.savefig(loss_path, format='pdf', bbox_inches='tight')
    plt.show()
    print(f'저장: {loss_path}')

    # ── Confusion Matrix 저장 (PDF, 제목 없음) ────────────────────────
    fig, axes = plt.subplots(1, n_folds, figsize=(4.5*n_folds, 3.8))
    if n_folds == 1:
        axes = [axes]
    for i, cm in enumerate(fold_cms):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['No Bulging', 'Bulging'],
                    yticklabels=['No Bulging', 'Bulging'],
                    ax=axes[i])
        axes[i].annotate(f'Fold {i+1}  (F1={fold_records[i]["f1"]:.4f})',
                         xy=(0.5, -0.18), xycoords='axes fraction',
                         ha='center', fontsize=9)
        axes[i].set_xlabel('Predicted')
        axes[i].set_ylabel('Actual')
    plt.tight_layout()
    cm_path = os.path.join(exp_dir, 'confusion_matrices.pdf')
    plt.savefig(cm_path, format='pdf', bbox_inches='tight')
    plt.show()
    print(f'저장: {cm_path}')

    return df_res, fold_curves

print('run_kfold_cv() 로드 완료 — 결과는 results/<exp_name>/ 에 저장됨')

In [ ]:
# ── Sec 13: 3-Fold CV 베이스라인 (bbox crop, SpiderDiscCNN_GAP) ──────────
from functools import partial

# Dataset 팩토리: Sec 6-1과 동일 (bbox crop + Resize 128×128, 정규화 없음)
def make_ds_61(samples):
    return SpiderDiscDataset(samples, bulging_labels, image_files_map, mask_files_map)

# 모델 팩토리: SpiderDiscCNN_GAP_b0
def make_model_61():
    return SpiderDiscCNN_GAP_b0()

# 환자 레벨 stratify 레이블
pid_list_cv = sorted(valid_pids_b)
strat_cv    = [
    int(any(bulging_labels.get((p, d), 0) == 1 for d in range(1, 7)))
    for p in pid_list_cv
]

df_13_cv, fold_curves_13 = run_kfold_cv(
    exp_name      = '13_cv_baseline',
    pid_list      = pid_list_cv,
    strat_labels  = strat_cv,
    all_samples   = all_samples_b,
    labels_dict   = bulging_labels,
    make_dataset_fn = make_ds_61,
    make_model_fn   = make_model_61,
    n_folds     = 3,
    max_epochs  = 50,
    patience    = 10,
    batch_size  = 32,
    lr          = 1e-4,
    seed        = 0,
    log_every   = 10,
)

# 6. 성능 개선 실험

3-Fold CV(F1=0.7613)를 베이스라인으로, 전처리 개선을 단계적으로 적용한다.

| 실험 | 전처리 | 입력 | CV Mean F1 |
|------|--------|------|-----------|
| 6-1 (Sec 14) | 등방 리샘플 0.75mm/px | 1ch 128×128 | 0.7531 ± 0.014 |
| 6-2 (Sec 17) | N4 보정 + Foreground Norm | 1ch 128×128 | 0.7620 ± 0.026 |
| 6-3 (Sec 16) | 등방 리샘플 + L/C/R 3채널 | 3ch 128×128 | **0.7665 ± 0.027** |

## 6-1. 등방 리샘플링 (Sec 14)

픽셀 크기가 환자마다 달라 물리적 스케일이 불일치하는 문제 해결.
`scipy.ndimage.zoom`으로 모든 이미지를 0.75mm/px로 등방 리샘플 후 디스크 중심 크롭.

In [ ]:
from scipy.ndimage import zoom as ndimage_zoom

# TARGET_SP, OUT_SIZE: Sec 8 셀에서 정의 (0.75 mm/px, 128 px)
# 커널 재시작 시를 대비해 여기서도 선언
_TARGET_SP = 0.75  # mm/px
_OUT_SIZE  = 128   # px → 물리 시야 96mm × 96mm

# ── Dataset (Sec 8 전처리 방식) ────────────────────────────────────
class SpiderDiscDatasetIso(Dataset):
    """
    Sec 8 전처리 적용 버전:
      1) mid-sagittal 슬라이스 추출
      2) scipy.ndimage.zoom → _TARGET_SP 등방 리샘플
      3) 디스크 중심 기준 _OUT_SIZE×_OUT_SIZE 중심 크롭 + 제로패딩(검정)
    정규화·증강 없음.
    """
    def __init__(self, samples, labels, img_files_map, mask_files_map,
                 target_sp=_TARGET_SP, out_size=_OUT_SIZE):
        self.samples        = samples
        self.labels         = labels
        self.img_files_map  = img_files_map
        self.mask_files_map = mask_files_map
        self.target_sp      = target_sp
        self.out_size       = out_size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, disc_idx = self.samples[idx]
        sitk_img  = sitk.ReadImage(self.img_files_map[f'{pid}_t2'])
        sitk_mask = sitk.ReadImage(self.mask_files_map[f'{pid}_t2'])
        img_arr   = sitk.GetArrayFromImage(sitk_img).astype(np.float32)
        mask_arr  = sitk.GetArrayFromImage(sitk_mask)
        sp        = sitk_img.GetSpacing()  # (sx, sy, sz)

        # mid-sagittal 슬라이스 추출
        ax  = int(np.argmin(img_arr.shape))
        mid = img_arr.shape[ax] // 2
        if ax == 0:
            img_2d, mask_2d = img_arr[mid], mask_arr[mid]
            sh, sw = sp[1], sp[0]
        elif ax == 1:
            img_2d, mask_2d = img_arr[:, mid, :], mask_arr[:, mid, :]
            sh, sw = sp[2], sp[0]
        else:
            img_2d, mask_2d = img_arr[:, :, mid], mask_arr[:, :, mid]
            sh, sw = sp[2], sp[1]

        # 등방 리샘플링 → target_sp
        tsp  = self.target_sp
        zh   = sh / tsp if sh > 0 else 1.0
        zw   = sw / tsp if sw > 0 else 1.0
        img_rs  = ndimage_zoom(img_2d,  (zh, zw), order=1)
        disc_rs = ndimage_zoom(
            (mask_2d == (200 + disc_idx)).astype(np.float32),
            (zh, zw), order=0
        ) > 0.5

        # 디스크 중심 계산 (리샘플된 공간)
        rows = np.where(disc_rs.any(axis=1))[0]
        cols = np.where(disc_rs.any(axis=0))[0]
        if len(rows) == 0:
            cr, cc = img_rs.shape[0] // 2, img_rs.shape[1] // 2
        else:
            cr = int((rows[0] + rows[-1]) // 2)
            cc = int((cols[0] + cols[-1]) // 2)

        # 중심 크롭 + 제로패딩
        half = self.out_size // 2
        H, W = img_rs.shape
        result = np.zeros((self.out_size, self.out_size), dtype=np.float32)
        r0 = cr - half;  r1 = r0 + self.out_size
        c0 = cc - half;  c1 = c0 + self.out_size
        sr0, sr1 = max(0, r0), min(H, r1)
        sc0, sc1 = max(0, c0), min(W, c1)
        dr0 = sr0 - r0;  dr1 = dr0 + (sr1 - sr0)
        dc0 = sc0 - c0;  dc1 = dc0 + (sc1 - sc0)
        result[dr0:dr1, dc0:dc1] = img_rs[sr0:sr1, sc0:sc1]

        img_t = torch.from_numpy(result).unsqueeze(0)  # (1, 128, 128)
        return img_t, torch.tensor([self.labels[(pid, disc_idx)]], dtype=torch.float32)


def make_ds_14(samples):
    return SpiderDiscDatasetIso(samples, bulging_labels, image_files_map, mask_files_map)

def make_model_14():
    return SpiderDiscCNN_GAP_b0()

# 동작 확인
ds_check = SpiderDiscDatasetIso(all_samples_b[:1], bulging_labels, image_files_map, mask_files_map)
img_c, lbl_c = ds_check[0]
print(f'샘플 shape: {tuple(img_c.shape)}, label: {lbl_c.item()}')
assert img_c.shape == (1, 128, 128), f'shape 불일치: {img_c.shape}'
print('SpiderDiscDatasetIso 정의 완료')


In [ ]:
# ── Sec 14 전처리 확인: 샘플 크롭 시각화 ──────────────────────────
import random, matplotlib.pyplot as plt

random.seed(42)
sample_items = random.sample(all_samples_b, 12)
ds_viz = SpiderDiscDatasetIso(sample_items, bulging_labels, image_files_map, mask_files_map)

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    img_t, lbl = ds_viz[i]
    pid, disc_idx = sample_items[i]
    ax.imshow(img_t.squeeze().numpy(), cmap='gray')
    ax.axis('off')
    ax.set_title(f'P{pid} D{disc_idx}\n{"Bulging" if lbl.item()==1 else "No Bulging"}',
                 fontsize=7)

plt.tight_layout()
plt.show()
print(f'FOV: {_OUT_SIZE * _TARGET_SP:.0f}mm × {_OUT_SIZE * _TARGET_SP:.0f}mm @ {_TARGET_SP}mm/px')


In [ ]:
# ── CV 실행 ──────────────────────────────────────────────────────────
df_14_cv, fold_curves_14 = run_kfold_cv(
    exp_name        = '14_cv_iso',
    pid_list        = pid_list_cv,
    strat_labels    = strat_cv,
    all_samples     = all_samples_b,
    labels_dict     = bulging_labels,
    make_dataset_fn = make_ds_14,              
    make_model_fn   = make_model_14,
    n_folds    = 3,
    max_epochs = 200,
    patience   = 10,
    batch_size = 32,
    lr         = 1e-4,
    seed       = 0,
    log_every  = 10,
)


## 6-2. N4 편향 보정 + Foreground 정규화 (Sec 17)

MRI 코일 특성에 의한 신호 불균일을 N4ITK Bias Field Correction으로 보정.
크롭 내 전경(≠0) 픽셀 기준 mean/std 정규화 추가.
증강 없이 전처리 개선만으로 효과 측정.

In [ ]:
import os
from tqdm.auto import tqdm

N4_DIR = 'data/n4_2d'
os.makedirs(N4_DIR, exist_ok=True)

corrector = sitk.N4BiasFieldCorrectionImageFilter()

for pid in tqdm(sorted(valid_pids_b), desc='N4 Bias Correction'):
    out_path = os.path.join(N4_DIR, f'{pid}.npz')
    if os.path.exists(out_path):
        continue  # 이미 계산된 경우 skip

    sitk_img = sitk.ReadImage(image_files_map[f'{pid}_t2'])
    img_arr  = sitk.GetArrayFromImage(sitk_img).astype(np.float32)
    sp       = sitk_img.GetSpacing()  # (sx, sy, sz)

    # mid-sagittal 슬라이스 추출
    ax  = int(np.argmin(img_arr.shape))
    mid = img_arr.shape[ax] // 2
    if ax == 0:
        img_2d = img_arr[mid];          sh, sw = sp[1], sp[0]
    elif ax == 1:
        img_2d = img_arr[:, mid, :];    sh, sw = sp[2], sp[0]
    else:
        img_2d = img_arr[:, :, mid];    sh, sw = sp[2], sp[1]

    # SimpleITK 2D 이미지 생성
    sitk_2d = sitk.Cast(sitk.GetImageFromArray(img_2d), sitk.sitkFloat32)
    sitk_2d.SetSpacing([float(sw), float(sh)])

    # Otsu 마스크 (배경 제외 → N4 수렴 안정화)
    mask = sitk.OtsuThreshold(sitk_2d, 0, 1, 200)

    # N4 보정 실행
    corrected     = corrector.Execute(sitk_2d, mask)
    corrected_arr = sitk.GetArrayFromImage(corrected)

    # 캐시 저장 (sh, sw 함께 저장 → Dataset이 3D 재로드 불필요)
    np.savez_compressed(out_path,
                        arr=corrected_arr,
                        sh=np.array([sh]),
                        sw=np.array([sw]))

print(f'완료: {N4_DIR}/ 내 {len(os.listdir(N4_DIR))}개 파일')


In [ ]:
import torchvision.transforms as T

# ── Train-only 증강 파이프라인 ─────────────────────────────────────
_train_aug_15 = T.Compose([
    T.RandomAffine(
        degrees=10,              # 회전 ±10°
        translate=(0.08, 0.08),  # 이동 ±8% (≈ ±10px)
    ),
    T.ElasticTransform(
        alpha=25.0,  # 변형 강도
        sigma=4.0,   # 변형 부드러움
    ),
])

def _add_gaussian_noise(img_t, std=0.02):
    return img_t + torch.randn_like(img_t) * std


# ── Dataset 클래스 ──────────────────────────────────────────────────
class SpiderDiscDatasetN4Norm(Dataset):
    """
    Sec 15 전처리:
      1) data/n4_2d/{pid}.npz 에서 N4 보정 2D 슬라이스 로드
      2) scipy.ndimage.zoom → 0.75mm/px 등방 리샘플
      3) 디스크 중심 기준 128×128 중심 크롭 + 제로패딩
      4) foreground(≠0) 픽셀 mean/std → 전체 크롭 정규화
      5) [Train only] RandomAffine + ElasticTransform + Gaussian Noise
    """
    def __init__(self, samples, labels, n4_dir, mask_files_map,
                 target_sp=_TARGET_SP, out_size=_OUT_SIZE, augment=False):
        self.samples        = samples
        self.labels         = labels
        self.n4_dir         = n4_dir
        self.mask_files_map = mask_files_map
        self.target_sp      = target_sp
        self.out_size       = out_size
        self.augment        = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, disc_idx = self.samples[idx]

        # 1. N4 보정 슬라이스 + spacing 로드
        data   = np.load(os.path.join(self.n4_dir, f'{pid}.npz'))
        img_2d = data['arr'].astype(np.float32)
        sh, sw = float(data['sh'][0]), float(data['sw'][0])

        # 2. 마스크에서 disc 위치 추출
        sitk_mask = sitk.ReadImage(self.mask_files_map[f'{pid}_t2'])
        mask_arr  = sitk.GetArrayFromImage(sitk_mask)
        ax  = int(np.argmin(mask_arr.shape))
        mid = mask_arr.shape[ax] // 2
        if ax == 0:   mask_2d = mask_arr[mid]
        elif ax == 1: mask_2d = mask_arr[:, mid, :]
        else:         mask_2d = mask_arr[:, :, mid]

        # 3. 등방 리샘플링 → 0.75mm/px
        tsp    = self.target_sp
        zh, zw = (sh / tsp if sh > 0 else 1.0), (sw / tsp if sw > 0 else 1.0)
        img_rs  = ndimage_zoom(img_2d, (zh, zw), order=1)
        disc_rs = ndimage_zoom(
            (mask_2d == (200 + disc_idx)).astype(np.float32),
            (zh, zw), order=0
        ) > 0.5

        # 4. 디스크 중심 계산 + 중심 크롭 + 제로패딩
        rows = np.where(disc_rs.any(axis=1))[0]
        cols = np.where(disc_rs.any(axis=0))[0]
        cr = int((rows[0]+rows[-1])//2) if len(rows) else img_rs.shape[0]//2
        cc = int((cols[0]+cols[-1])//2) if len(cols) else img_rs.shape[1]//2

        half = self.out_size // 2
        H, W = img_rs.shape
        crop = np.zeros((self.out_size, self.out_size), dtype=np.float32)
        r0, c0 = cr - half, cc - half
        sr0, sr1 = max(0, r0), min(H, r0 + self.out_size)
        sc0, sc1 = max(0, c0), min(W, c0 + self.out_size)
        dr0 = sr0 - r0;  dr1 = dr0 + (sr1 - sr0)
        dc0 = sc0 - c0;  dc1 = dc0 + (sc1 - sc0)
        crop[dr0:dr1, dc0:dc1] = img_rs[sr0:sr1, sc0:sc1]

        # 5. Foreground(≠0) 기반 정규화
        fg = crop[crop != 0]
        if len(fg) > 0 and fg.std() > 1e-8:
            crop = (crop - fg.mean()) / fg.std()

        img_t = torch.from_numpy(crop).unsqueeze(0)  # (1, 128, 128)

        # 6. [Train only] 증강
        if self.augment:
            img_t = _train_aug_15(img_t)
            img_t = _add_gaussian_noise(img_t, std=0.02)

        return img_t, torch.tensor([self.labels[(pid, disc_idx)]], dtype=torch.float32)


def make_train_ds_15(samples):
    return SpiderDiscDatasetN4Norm(
        samples, bulging_labels, N4_DIR, mask_files_map, augment=True)

def make_val_ds_15(samples):
    return SpiderDiscDatasetN4Norm(
        samples, bulging_labels, N4_DIR, mask_files_map, augment=False)

def make_model_15():
    return SpiderDiscCNN_GAP_b0()

# 동작 확인
ds_check15 = make_val_ds_15(all_samples_b[:1])
img_c, lbl_c = ds_check15[0]
print(f'샘플 shape: {tuple(img_c.shape)}, label: {lbl_c.item()}')
assert img_c.shape == (1, 128, 128), f'shape 불일치: {img_c.shape}'
# 증강 확인: 같은 인덱스 두 번 호출 시 달라야 함
ds_aug_check = make_train_ds_15(all_samples_b[:1])
a, b = ds_aug_check[0][0], ds_aug_check[0][0]
print(f'증강 확인 (Train): 두 호출이 다른가? {not torch.equal(a, b)}')
print('SpiderDiscDatasetN4Norm 정의 완료')


In [ ]:
# Sec 17: N4 + Foreground Norm, 증강 없음
# SpiderDiscDatasetN4Norm은 Cell 118에서 이미 정의됨

def make_ds_17(samples):
    return SpiderDiscDatasetN4Norm(
        samples, bulging_labels, N4_DIR, mask_files_map, augment=False)

def make_model_17():
    return SpiderDiscCNN_GAP_b0()

# 동작 확인
ds_check17 = make_ds_17(all_samples_b[:1])
img_c, lbl_c = ds_check17[0]
assert img_c.shape == (1, 128, 128), f'shape 불일치: {img_c.shape}'
print(f'샘플 shape: {tuple(img_c.shape)}, label: {lbl_c.item()}')
print('Sec 17 dataset 정의 완료')

In [ ]:
df_17_cv, fold_curves_17 = run_kfold_cv(
    exp_name             = '17_cv_n4_noaug',
    pid_list             = pid_list_cv,
    strat_labels         = strat_cv,
    all_samples          = all_samples_b,
    labels_dict          = bulging_labels,
    make_dataset_fn      = make_ds_17,
    make_model_fn        = make_model_17,
    n_folds    = 3,
    max_epochs = 200,
    patience   = 10,
    batch_size = 32,
    lr         = 1e-4,
    seed       = 0,
    log_every  = 10,
)

## 6-3. 다중 슬라이스 3채널 입력 (Sec 16)

중앙 시상면 1장만 보는 방식은 편측 팽윤을 놓칠 수 있음.
물리적으로 ±4mm 떨어진 좌(L)·중(C)·우(R) 시상면 3장을 3채널로 stack.
모델의 첫 번째 Conv2d 입력만 1→3채널로 확장 (파라미터 +576 = 19,457 total).

In [ ]:
# ── Dataset: 좌·중·우 3채널 ─────────────────────────────────────
class SpiderDiscDataset3Ch(Dataset):
    """등방 리샘플(0.75mm/px) + 중심 크롭 + 좌·중·우 3채널 (3, 128, 128)"""
    def __init__(self, samples, labels, img_files_map, mask_files_map,
                 target_sp=_TARGET_SP, out_size=_OUT_SIZE):
        self.samples        = samples
        self.labels         = labels
        self.img_files_map  = img_files_map
        self.mask_files_map = mask_files_map
        self.target_sp      = target_sp
        self.out_size       = out_size

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _get_slice(arr, ax, idx):
        if ax == 0:   return arr[idx]
        elif ax == 1: return arr[:, idx, :]
        else:         return arr[:, :, idx]

    @staticmethod
    def _center_crop(img_rs, cr, cc, out_size):
        half = out_size // 2
        H, W = img_rs.shape
        result = np.zeros((out_size, out_size), dtype=np.float32)
        r0, c0 = cr - half, cc - half
        sr0, sr1 = max(0, r0), min(H, r0 + out_size)
        sc0, sc1 = max(0, c0), min(W, c0 + out_size)
        dr0 = sr0 - r0;  dr1 = dr0 + (sr1 - sr0)
        dc0 = sc0 - c0;  dc1 = dc0 + (sc1 - sc0)
        result[dr0:dr1, dc0:dc1] = img_rs[sr0:sr1, sc0:sc1]
        return result

    def __getitem__(self, idx):
        pid, disc_idx = self.samples[idx]
        si  = sitk.ReadImage(self.img_files_map[f'{pid}_t2'])
        sm  = sitk.ReadImage(self.mask_files_map[f'{pid}_t2'])
        img = sitk.GetArrayFromImage(si).astype(np.float32)
        msk = sitk.GetArrayFromImage(sm)
        sp  = si.GetSpacing()  # (sx, sy, sz)

        ax  = int(np.argmin(img.shape))
        mid = img.shape[ax] // 2
        if ax == 0:   sh, sw, tsp_sl = sp[1], sp[0], sp[2]
        elif ax == 1: sh, sw, tsp_sl = sp[2], sp[0], sp[1]
        else:         sh, sw, tsp_sl = sp[2], sp[1], sp[0]

        # 좌·중·우 슬라이스 인덱스 (물리적 ±4mm)
        N      = img.shape[ax]
        offset = max(1, round(4.0 / tsp_sl))
        idx_L  = max(0,   mid - offset)
        idx_R  = min(N-1, mid + offset)

        # 등방 리샘플 zoom 비율
        tsp = self.target_sp
        zh  = sh / tsp if sh > 0 else 1.0
        zw  = sw / tsp if sw > 0 else 1.0

        # 중앙 슬라이스 마스크로 디스크 중심 계산
        msk_c  = self._get_slice(msk, ax, mid)
        dm_rs  = ndimage_zoom((msk_c == (200 + disc_idx)).astype(np.float32),
                              (zh, zw), order=0) > 0.5
        rows   = np.where(dm_rs.any(axis=1))[0]
        cols   = np.where(dm_rs.any(axis=0))[0]
        img_cr = ndimage_zoom(self._get_slice(img, ax, mid), (zh, zw), order=1)
        cr = int((rows[0]+rows[-1])//2) if len(rows) else img_cr.shape[0]//2
        cc = int((cols[0]+cols[-1])//2) if len(cols) else img_cr.shape[1]//2

        # L/C/R 각각 리샘플 + 동일 좌표 중심 크롭
        channels = []
        for sl_idx in [idx_L, mid, idx_R]:
            rs    = ndimage_zoom(self._get_slice(img, ax, sl_idx), (zh, zw), order=1)
            patch = self._center_crop(rs, cr, cc, self.out_size)
            channels.append(patch)

        img_t = torch.from_numpy(np.stack(channels, axis=0))  # (3, 128, 128)
        return img_t, torch.tensor([self.labels[(pid, disc_idx)]], dtype=torch.float32)


# ── Model: 첫 번째 Conv 입력 채널 1 → 3 ─────────────────────────────
class SpiderDiscCNN_GAP_3ch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, x):
        return self.head(self.gap(self.features(x)))


def make_ds_16(samples):
    return SpiderDiscDataset3Ch(samples, bulging_labels, image_files_map, mask_files_map)

def make_model_16():
    return SpiderDiscCNN_GAP_3ch()

# 동작 확인
ds_check16 = make_ds_16(all_samples_b[:1])
img_c, lbl_c = ds_check16[0]
print(f'샘플 shape: {tuple(img_c.shape)}, label: {lbl_c.item()}')
assert img_c.shape == (3, 128, 128), f'shape 불일치: {img_c.shape}'
total_params = sum(p.numel() for p in SpiderDiscCNN_GAP_3ch().parameters())
print(f'모델 파라미터: {total_params:,}')
print('SpiderDiscDataset3Ch + SpiderDiscCNN_GAP_3ch 정의 완료')


In [ ]:
# ── Sec 16 전처리 확인: L/C/R 3채널 시각화 ───────────────────────
import random, matplotlib.pyplot as plt

random.seed(42)
sample_items_16 = random.sample(all_samples_b, 4)
ds_viz_16 = make_ds_16(sample_items_16)

row_labels = ['Left', 'Center', 'Right']
fig, axes = plt.subplots(3, 4, figsize=(11, 8))

for col, (pid, disc_idx) in enumerate(sample_items_16):
    img_t, lbl = ds_viz_16[col]
    label_str = 'Bulging' if lbl.item() == 1 else 'No Bulging'
    for row in range(3):
        ax = axes[row, col]
        ax.imshow(img_t[row].numpy(), cmap='gray', aspect='equal')
        ax.axis('off')
        if row == 0:
            ax.set_title(f'P{pid} D{disc_idx}\n{label_str}', fontsize=7.5)
    axes[0, col].set_ylabel('', visible=False)

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=9)
    axes[row, 0].yaxis.set_visible(True)
    axes[row, 0].tick_params(left=False, labelleft=False)

plt.tight_layout(h_pad=0.4, w_pad=0.3)
plt.show()
print(f'FOV: {_OUT_SIZE * _TARGET_SP:.0f}mm × {_OUT_SIZE * _TARGET_SP:.0f}mm | offset = ±4mm (through-plane)')


In [ ]:
# ── CV 실행 ──────────────────────────────────────────────────────────
df_16_cv, fold_curves_16 = run_kfold_cv(
    exp_name        = '16_cv_3ch',
    pid_list        = pid_list_cv,
    strat_labels    = strat_cv,
    all_samples     = all_samples_b,
    labels_dict     = bulging_labels,
    make_dataset_fn = make_ds_16,
    make_model_fn   = make_model_16,
    n_folds    = 3,
    max_epochs = 200,
    patience   = 10,
    batch_size = 32,
    lr         = 1e-4,
    seed       = 0,
    log_every  = 10,
)


# 7. 실험 결과 요약

| Section | 핵심 변경 | CV Mean F1 |
|---------|-----------|-----------|
| 5-2 (Sec 13) | 3-Fold CV 베이스라인 (bbox crop) | 0.7613 ± 0.020 |
| 6-1 (Sec 14) | 등방 리샘플 0.75mm/px | 0.7531 ± 0.014 |
| 6-2 (Sec 17) | N4 보정 + Foreground Norm | **0.7620 ± 0.026** |
| 6-3 (Sec 16) | 3채널 다중 슬라이스 | **0.7665 ± 0.027** |

등방 리샘플 단독은 소폭 하락했으나, N4+Norm 추가 시 베이스라인을 상회.
3채널 입력이 전체 실험 중 최고 성능(F1=0.7665)을 달성했다.